# Robustness Experiments — Attacks & Defenses

Tests the federated PIDL model against data-poisoning attacks and the
update-clipping defense. Results are saved under `results_robustness/`.

**Run after notebook 01** (datasets are already downloaded and cached).

| Experiment | Description |
|---|---|
| Clean baseline | No attack, no defense |
| Gaussian noise | 1 malicious client adds Gaussian noise to images |
| Label flip | 1 malicious client flips 30 % of labels |
| Noise + clipping | Gaussian noise attack + update-clipping defense |
| Label flip + clipping | Label-flip attack + update-clipping defense |

---
## § 1 — Setup

In [ ]:
# ── Edit your GitHub repo URL here ───────────────────────────────────
GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'
# ──────────────────────────────────────────────────────────────────────

import gc, json, os, sys, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

PROJECT_ROOT = Path('/content/medical_fl_pidl')
if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
else:
    os.system(f'git -C {PROJECT_ROOT} pull')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

RESULTS_ROOT = PROJECT_ROOT / 'results_robustness'
FIGURES_DIR  = RESULTS_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt

plt.rcParams.update({
    'font.family':'serif','font.size':11,'axes.labelsize':12,
    'axes.titlesize':13,'figure.dpi':150,'savefig.dpi':300,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,
    'figure.constrained_layout.use':True,
})

WONG = ['#E69F00','#56B4E9','#009E73','#F0E442','#0072B2','#D55E00','#CC79A7','#000000']

def savefig(name):
    p = FIGURES_DIR / f'{name}.png'
    plt.savefig(p, bbox_inches='tight', dpi=300)
    print(f'  Saved: {p.name}')

print('Setup complete.')


---
## § 2 — Dataset & Experiment Config

In [ ]:
# Choose ONE dataset for robustness experiments.
# The data root must already be downloaded (notebook 01 caches it).
DATASET_NAME = 'brain_tumor_mri'   # or 'colon_cancer_or_pathology' / 'covid'
DATA_ROOT    = '/root/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/1/Training'

# FL settings (keep small for speed)
NUM_CLIENTS   = 3
NUM_ROUNDS    = 5
LOCAL_EPOCHS  = 2

# Attack settings
MALICIOUS_IDS = [0]       # client 0 is the attacker in all attack experiments
NOISE_STD     = 0.8       # Gaussian noise std (normalised pixel space)
FLIP_PROB     = 0.30      # fraction of labels to flip

# Defense settings
CLIP_NORM     = 3.0       # max L2 norm of client update

print(f'Dataset : {DATASET_NAME}')
print(f'Data root : {DATA_ROOT}')
print(f'Clients : {NUM_CLIENTS}  |  Rounds : {NUM_ROUNDS}  |  Local epochs : {LOCAL_EPOCHS}')
print(f'Attacker clients : {MALICIOUS_IDS}  |  noise_std={NOISE_STD}  flip_prob={FLIP_PROB}')
print(f'Clip norm : {CLIP_NORM}')


---
## § 3 — Experiment Runner

In [ ]:
from flwr.simulation import run_simulation
from federated.server_app import app as _server_app
from federated.client_app import _make_client_app

def run_robustness_experiment(
    experiment_name: str,
    enable_attack:          bool  = False,
    attack_type:            str   = 'gaussian_noise',
    enable_update_clipping: bool  = False,
) -> dict:
    """Run one robustness experiment and return a summary dict."""
    log_dir = RESULTS_ROOT / DATASET_NAME / experiment_name
    log_dir.mkdir(parents=True, exist_ok=True)

    run_cfg = {
        # Core FL
        'dataset_name':      DATASET_NAME,
        'data_root':         DATA_ROOT,
        'log_dir':           str(log_dir),
        'num_clients':       NUM_CLIENTS,
        'num_server_rounds': NUM_ROUNDS,
        'min_fit_clients':   NUM_CLIENTS,
        'local_epochs':      LOCAL_EPOCHS,
        'batch_size':        32,
        'learning_rate':     0.001,
        'image_size':        224,
        'feature_layer':     'layer2',
        'regularizer_type':  'perona_malik',
        'lambda_pm':         0.1,
        'use_grid_loss':     True,
        'grid_size':         4,
        'k':                 1.0,
        'random_seed':       42,
        'use_secagg':        False,
        # Robustness
        'enable_attack':            enable_attack,
        'attack_type':              attack_type,
        'malicious_client_ids':     ','.join(str(i) for i in MALICIOUS_IDS),
        'noise_std':                NOISE_STD,
        'label_flip_probability':   FLIP_PROB,
        'enable_update_clipping':   enable_update_clipping,
        'clip_norm':                CLIP_NORM,
    }

    # Write config
    (log_dir / 'config.json').write_text(json.dumps(run_cfg, indent=2))

    os.environ['FL_RUN_OVERRIDE'] = json.dumps(run_cfg)
    _client_app = _make_client_app()

    gpu_frac = 0.5 if torch.cuda.is_available() else 0.0
    backend_cfg = {'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}}

    print(f'\n--- {experiment_name} ---')
    t0 = time.time()
    try:
        run_simulation(
            server_app    =_server_app,
            client_app    =_client_app,
            num_supernodes=NUM_CLIENTS,
            backend_config=backend_cfg,
        )
        elapsed = time.time() - t0
        print(f'  Done in {elapsed:.0f}s')
    except Exception as exc:
        print(f'  FAILED: {exc}')
        import traceback; traceback.print_exc()
        return {'experiment': experiment_name, 'error': str(exc)}
    finally:
        os.environ.pop('FL_RUN_OVERRIDE', None)

    # Build summary from fl_rounds.csv
    csv_path = log_dir / 'fl_rounds.csv'
    jsonl_path = log_dir / 'round_metrics.jsonl'
    summary = {'experiment': experiment_name, 'elapsed_sec': round(elapsed, 1)}

    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df = df[df['round'] > 0]
        if not df.empty:
            last = df.iloc[-1]
            summary.update({
                'final_accuracy':    round(float(last['global_test_acc']),  4),
                'best_accuracy':     round(float(df['global_test_acc'].max()), 4),
                'final_macro_f1':    round(float(last['f1_macro']),         4),
                'final_ece':         round(float(last['ece']),              4),
                'final_loss':        round(float(last['global_test_loss']), 4),
            })

    # Pull secagg / training times and ce/reg losses from JSONL
    if jsonl_path.exists():
        lines = jsonl_path.read_text().strip().splitlines()
        recent = [json.loads(l) for l in lines[-NUM_ROUNDS:]]
        summary['secagg_overhead_mean_sec'] = round(
            sum(r.get('secagg_overhead_sec',0) for r in recent) / max(len(recent),1), 4)
        summary['train_ce_loss_final']  = round(recent[-1].get('train_ce_loss', 0), 4)  if recent else 0
        summary['train_reg_loss_final'] = round(recent[-1].get('train_pidl_loss', 0), 4) if recent else 0

    (log_dir / 'robustness_summary.json').write_text(json.dumps(summary, indent=2))
    print(f'  acc={summary.get("final_accuracy","?")}')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

print('run_robustness_experiment() defined.')


---
## § 4 — Run All 5 Experiments

In [ ]:
EXPERIMENTS = [
    # (name,              enable_attack, attack_type,      enable_clipping)
    ('clean',             False,         'gaussian_noise', False),
    ('noisy_client',      True,          'gaussian_noise', False),
    ('label_flip',        True,          'label_flip',     False),
    ('noisy_clipped',     True,          'gaussian_noise', True),
    ('label_flip_clipped',True,          'label_flip',     True),
]

all_results = []
for name, atk, atype, clip in EXPERIMENTS:
    result = run_robustness_experiment(
        experiment_name        =name,
        enable_attack          =atk,
        attack_type            =atype,
        enable_update_clipping =clip,
    )
    all_results.append(result)

results_df = pd.DataFrame(all_results).set_index('experiment')
results_df.to_csv(RESULTS_ROOT / 'robustness_comparison.csv')
print('\nAll experiments complete.')
print(results_df[['final_accuracy','best_accuracy','final_macro_f1','final_ece']].to_string())


---
## § 5 — Plots

In [ ]:
# ── Load per-round data for training curves ───────────────────────────
EXP_COLORS = {
    'clean':              WONG[1],
    'noisy_client':       WONG[5],
    'label_flip':         WONG[0],
    'noisy_clipped':      WONG[2],
    'label_flip_clipped': WONG[6],
}
EXP_LS = {
    'clean':'-','noisy_client':'--','label_flip':':',
    'noisy_clipped':'-.','label_flip_clipped':(0,(3,1,1,1)),
}
LABEL = {
    'clean':'Clean baseline',
    'noisy_client':'Gaussian noise attack',
    'label_flip':'Label-flip attack',
    'noisy_clipped':'Noise + clipping',
    'label_flip_clipped':'Label-flip + clipping',
}

per_round = {}
for name, *_ in EXPERIMENTS:
    csv = RESULTS_ROOT / DATASET_NAME / name / 'fl_rounds.csv'
    if csv.exists():
        df = pd.read_csv(csv)
        per_round[name] = df[df['round'] > 0].reset_index(drop=True)

# ── Plot 1: Accuracy & F1 over rounds ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'{DATASET_NAME.replace("_"," ").title()} — Attack/Defense Comparison', fontweight='bold')

for exp, df in per_round.items():
    c  = EXP_COLORS.get(exp, 'gray')
    ls = EXP_LS.get(exp, '-')
    lb = LABEL.get(exp, exp)
    axes[0].plot(df['round'], df['global_test_acc'], color=c, ls=ls, marker='o', markersize=4, label=lb)
    axes[1].plot(df['round'], df['f1_macro'],        color=c, ls=ls, marker='o', markersize=4, label=lb)

for ax, title in zip(axes, ['Test Accuracy', 'Macro F1']):
    ax.set_xlabel('Round'); ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)

savefig('01_accuracy_f1_curves')
plt.show()

# ── Plot 2: Final metrics bar chart ───────────────────────────────────
if not results_df.empty and 'final_accuracy' in results_df.columns:
    metrics_to_plot = ['final_accuracy','best_accuracy','final_macro_f1','final_ece']
    available = [m for m in metrics_to_plot if m in results_df.columns]
    fig, axes = plt.subplots(1, len(available), figsize=(4.5*len(available), 5))
    if len(available) == 1: axes = [axes]
    fig.suptitle('Final Metrics by Experiment', fontweight='bold')

    for ax, metric in zip(axes, available):
        vals = results_df[metric].dropna()
        colors = [EXP_COLORS.get(i, 'gray') for i in vals.index]
        bars = ax.bar(range(len(vals)), vals.values, color=colors, edgecolor='white')
        ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
        ax.set_title(metric.replace('_',' ').title(), fontweight='bold')
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels([LABEL.get(i,i).replace(' ','\n') for i in vals.index], fontsize=7)
        ax.set_ylim(0, max(vals.max()*1.15, 0.1))
        ax.set_axisbelow(True)

    savefig('02_final_metrics_bar')
    plt.show()

# ── Plot 3: Training loss (CE vs PIDL reg) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Training Loss Decomposition', fontweight='bold')

for exp in per_round:
    jsonl = RESULTS_ROOT / DATASET_NAME / exp / 'round_metrics.jsonl'
    if not jsonl.exists(): continue
    lines = jsonl.read_text().strip().splitlines()
    recs  = [json.loads(l) for l in lines[-NUM_ROUNDS:]]
    rounds   = [r['round'] for r in recs]
    ce_loss  = [r.get('train_ce_loss', 0) for r in recs]
    reg_loss = [r.get('train_pidl_loss', 0) for r in recs]
    c = EXP_COLORS.get(exp, 'gray')
    ls = EXP_LS.get(exp, '-')
    lb = LABEL.get(exp, exp)
    axes[0].plot(rounds, ce_loss,  color=c, ls=ls, marker='o', markersize=3, label=lb)
    axes[1].plot(rounds, reg_loss, color=c, ls=ls, marker='o', markersize=3)

axes[0].set_title('CE Loss', fontweight='bold'); axes[0].set_xlabel('Round'); axes[0].set_ylabel('Loss')
axes[1].set_title('PIDL Reg Loss', fontweight='bold'); axes[1].set_xlabel('Round')
axes[0].legend(fontsize=8)
savefig('03_loss_decomposition')
plt.show()

print('All plots saved to', FIGURES_DIR)


---
## § 6 — Summary Table

In [ ]:
display_cols = ['final_accuracy','best_accuracy','final_macro_f1','final_ece',
                 'secagg_overhead_mean_sec','elapsed_sec']
available = [c for c in display_cols if c in results_df.columns]

styled = results_df[available].rename(index=LABEL)
styled.index.name = 'Experiment'

try:
    display(styled.style
        .format({c: '{:.4f}' for c in available}, na_rep='-')
        .background_gradient(cmap='Greens',
            subset=[c for c in available if 'accuracy' in c or 'f1' in c], vmin=0, vmax=1)
        .background_gradient(cmap='RdYlGn_r',
            subset=[c for c in available if 'ece' in c], vmin=0, vmax=0.5)
    )
except Exception:
    print(styled.to_string())


---
## § 7 — Download Results

In [ ]:
import subprocess
subprocess.run(['zip', '-r', '/content/robustness_results.zip',
                str(RESULTS_ROOT)], check=True)

from google.colab import files
files.download('/content/robustness_results.zip')
print('Download started.')
